# Week 11 Post-Class: Solutions

Reference solution showing one polished version of Idea 1 (Customer Support Assistant).

In [ ]:
import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    set_seed,
)
import json

set_seed(42)
# Runs on GPU, Apple Silicon (MPS), or CPU.
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

## Customer Support Assistant

**App:** Process customer support emails. Extract structured info, classify mood, draft response.

**Models:**
1. NER (extract product/account info)
2. Sentiment analysis (mood classification)
3. Phi-3-mini LLM (response generation)

In [ ]:
# Load all models
ner = pipeline('ner', aggregation_strategy='simple')
sentiment = pipeline('sentiment-analysis')

MODEL = 'microsoft/Phi-3-mini-4k-instruct'
llm_tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.float16 if DEVICE in ('cuda', 'mps') else torch.float32,  # v5: 'dtype' (was 'torch_dtype')
    device_map='auto' if DEVICE == 'cuda' else None,
    # No trust_remote_code: v5 has native Phi-3/Qwen support; the old bundled code crashes on v5.
)
if DEVICE != 'cuda':          # mps/cpu need an explicit move; cuda is placed by device_map='auto'
    llm_model = llm_model.to(DEVICE)

def llm(prompt, max_new_tokens=150, temperature=0.5):
    messages = [{'role': 'user', 'content': prompt}]
    # v5: apply_chat_template returns a BatchEncoding — use return_dict=True + generate(**inputs).
    inputs = llm_tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True, return_dict=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=0.9, do_sample=temperature > 0, pad_token_id=llm_tokenizer.eos_token_id
        )
    return llm_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
def process_support_email(email_text):
    """
    Process a customer support email through 3 models.
    Returns structured ticket + suggested response.
    """
    # Step 1: Extract entities (people, products, locations, dates)
    entities = ner(email_text)
    extracted = {}
    for ent in entities:
        ent_type = ent['entity_group']
        extracted.setdefault(ent_type, []).append(ent['word'])
    
    # Step 2: Classify sentiment
    sent = sentiment(email_text[:500])[0]
    
    # Step 3: Generate appropriate response based on sentiment
    if sent['label'] == 'NEGATIVE':
        prompt = (
            f'A customer wrote this complaint email:\n\n"{email_text}"\n\n'
            f'Write a brief, professional response that:\n'
            f'1. Acknowledges the issue\n'
            f'2. Apologizes sincerely\n'
            f'3. Offers a concrete next step\n\n'
            f'Maximum 4 sentences. Sign as "Customer Support".'
        )
    else:
        prompt = (
            f'A customer wrote this email:\n\n"{email_text}"\n\n'
            f'Write a brief, professional response that thanks them and addresses any questions. '
            f'Maximum 3 sentences. Sign as "Customer Support".'
        )
    
    response = llm(prompt, max_new_tokens=200, temperature=0.4)
    
    return {
        'priority': 'HIGH' if sent['label'] == 'NEGATIVE' and sent['score'] > 0.95 else 'NORMAL',
        'sentiment': sent['label'],
        'sentiment_confidence': float(sent['score']),
        'extracted_entities': extracted,
        'suggested_response': response,
    }

In [ ]:
# Test on 3 examples
test_emails = [
    'Hi, I ordered a Samsung TV from your store last Tuesday and it arrived broken. '
    'I have tried calling 3 times and no one has answered. This is unacceptable.',
    
    'Just wanted to say thank you for the quick delivery of my MacBook Pro! '
    'It arrived 2 days early and works perfectly. Great service from your team in Boston.',
    
    'Quick question: is it possible to upgrade my subscription from Basic to Premium '
    'before the next billing cycle? My account is john.smith@example.com.',
]

for i, email in enumerate(test_emails, 1):
    print(f'\n{"="*70}')
    print(f'EMAIL #{i}:')
    print(email)
    print(f'\nPROCESSED:')
    result = process_support_email(email)
    print(json.dumps(result, indent=2))

In [ ]:
# Edge cases
edge_cases = [
    '',  # empty
    'asdkfjhasdkfjhakjsdf',  # gibberish
    'Bonjour, j\'ai besoin d\'aide avec ma commande.',  # different language
    'A' * 5000,  # very long
]

for case in edge_cases:
    try:
        result = process_support_email(case)
        print(f'OK ({len(case)} chars): priority={result["priority"]}, sentiment={result["sentiment"]}')
    except Exception as e:
        print(f'FAIL ({len(case)} chars): {type(e).__name__}: {e}')

## What's good about this app

- Combines 3 different ML capabilities in one pipeline
- Output is structured (parseable JSON) — usable in downstream systems
- Sentiment-conditional response logic — different prompt for complaints vs other
- ~50 lines of meaningful code

## What's NOT production-ready

- **No input validation** — empty strings, gibberish would crash in real use
- **No prompt injection protection** — adversarial inputs could manipulate the LLM
- **No retry logic** for model failures
- **No monitoring** of output quality
- **Sentiment model assumes English** — non-English may misclassify
- **LLM may hallucinate facts** in responses (e.g., refund policies)
- **Response should be human-reviewed** before sending — never auto-send

## What to add for production

1. Input validation (length checks, language detection)
2. JSON schema validation on LLM output (or use structured output APIs)
3. Human-in-the-loop review before sending responses
4. Log all interactions for quality monitoring
5. Circuit breaker if LLM service fails
6. Rate limiting if exposed as API

**This is the difference between a POC and production.** The POC takes 1 hour. Production takes 6 months. The POC proves it's worth the 6 months.